# Features, Not Just Words: Toward CRFs

**Computational Text Analysis Pack · Module 16 — Sequence Labelling**

*Technique:* Feature engineering for sequence models; why CRFs replaced HMMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/16b_features_and_crfs.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

The HMM had one blind spot: it only knows a word by its identity. Meet a word
that was not in training and it has nothing to go on.

But an unknown word is not featureless. *Kandahar* is capitalised. *Hurriedly*
ends in *-ly*. *1897* is digits. **Conditional random fields** exist to use
exactly that: arbitrary, overlapping features of the token and its neighbours,
which an HMM's probability model cannot accommodate.

`sklearn-crfsuite` is not installed, so we build the feature-based version with
logistic regression over a sliding window — which is the same idea without the
sequence-level normalisation that makes a CRF a CRF.

## The tooling

**CRFsuite** (via `sklearn-crfsuite`) and **MALLET** are the classic CRF
implementations; **spaCy** and **Stanza** have replaced both for well-resourced
languages. The survey's point stands regardless of which you use: for sequence
labelling, **feature engineering is central and interpretable**, unlike the
learned features of a neural tagger.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import nltk

sentences = []
for passage in df_colonial["text"].head(400):
    for raw in str(passage).split("."):
        toks = tokenise(raw)
        if 5 <= len(toks) <= 30:
            sentences.append(toks)
SIMPLE = {"NN": "NOUN", "NNS": "NOUN", "NNP": "NOUN", "VB": "VERB",
          "VBD": "VERB", "VBG": "VERB", "VBN": "VERB", "VBP": "VERB",
          "VBZ": "VERB", "JJ": "ADJ", "RB": "ADV", "DT": "DET", "IN": "ADP",
          "PRP": "PRON", "CC": "CONJ", "CD": "NUM"}
data = [[(w, SIMPLE.get(t, "X")) for w, t in nltk.pos_tag(s)]
        for s in sentences[:1200]]
split = int(len(data) * 0.8)
train, test = data[:split], data[split:]
print(f"{len(train)} train / {len(test)} test sentences")

## The feature function

This is where the linguistics goes. Each feature is a claim about what predicts
a tag, and unlike a neural embedding you can read the list and argue with it.

In [ ]:
def features(words, i):
    w = words[i]
    f = {
        "word": w,
        "suffix3": w[-3:],
        "suffix2": w[-2:],
        "prefix2": w[:2],
        "is_digit": w.isdigit(),
        "has_hyphen": "-" in w,
        "length_band": min(len(w) // 3, 4),
        "prev": words[i - 1] if i else "<START>",
        "next": words[i + 1] if i + 1 < len(words) else "<END>",
        "prev_suffix2": words[i - 1][-2:] if i else "<START>",
    }
    return {f"{k}={v}" for k, v in f.items()}

demo = [w for w, _ in train[0]]
print("features for the word", repr(demo[2]), "in context:\n")
for feat in sorted(features(demo, 2)):
    print("   ", feat)

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def to_xy(dataset):
    X, y = [], []
    for sent in dataset:
        words = [w for w, _ in sent]
        for i, (_, tag) in enumerate(sent):
            X.append({f: 1 for f in features(words, i)})
            y.append(tag)
    return X, y

Xtr, ytr = to_xy(train)
Xte, yte = to_xy(test)
vec = DictVectorizer()
A = vec.fit_transform(Xtr)
B = vec.transform(Xte)
clf = LogisticRegression(max_iter=400, C=2.0).fit(A, ytr)
full = clf.score(B, yte)
print(f"{A.shape[1]:,} features, {A.shape[0]:,} training tokens")
print(f"accuracy with all features: {full:.3f}")

## Ablation: which features are doing the work?

Remove one feature group at a time and re-measure. This is how you justify a
feature set in a methods section, and it is the kind of claim a neural tagger
cannot support.

In [ ]:
GROUPS = ["word", "suffix3", "suffix2", "prefix2", "prev", "next",
          "is_digit", "length_band"]
rows = []
for drop in GROUPS:
    def limited(words, i, drop=drop):
        return {f for f in features(words, i) if not f.startswith(drop + "=")}
    X2, y2 = [], []
    for sent in train:
        ws = [w for w, _ in sent]
        for i, (_, tag) in enumerate(sent):
            X2.append({f: 1 for f in limited(ws, i)})
            y2.append(tag)
    Xt2 = []
    for sent in test:
        ws = [w for w, _ in sent]
        for i, _ in enumerate(sent):
            Xt2.append({f: 1 for f in limited(ws, i)})
    v2 = DictVectorizer()
    m = LogisticRegression(max_iter=400, C=2.0).fit(v2.fit_transform(X2), y2)
    acc = m.score(v2.transform(Xt2), yte)
    rows.append({"removed": drop, "accuracy": acc, "cost": full - acc})
abl = pd.DataFrame(rows).sort_values("cost", ascending=False)
print(abl.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(abl["removed"], abl["cost"], color="steelblue")
plt.axvline(0, color="black", lw=.8)
plt.xlabel("accuracy lost when this feature group is removed")
plt.title("Feature ablation")
plt.tight_layout()

print("\nUnknown-word handling, the reason features beat word identity:")
train_vocab = {w for sent in train for w, _ in sent}
unk_idx = [i for i, (w, _) in enumerate((wt for s in test for wt in s))
           if w not in train_vocab]
preds = clf.predict(B)
if unk_idx:
    hits = sum(1 for i in unk_idx if preds[i] == yte[i])
    print(f"   {len(unk_idx):,} unseen tokens, tagged correctly {hits/len(unk_idx):.3f}")
    print("   An HMM would have only the emission smoothing constant here.")

### 🔧 Your turn 1

Add a feature for whether the word is title-cased, re-run, and see where it lands in the ablation.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Capitalisation is usually one of the strongest single features for NOUN, and on
this corpus it will rank high — which is exactly the problem notebook 15a
raised. It is a feature of *English orthographic convention*, not of nouns, and
a tagger leaning on it will not transfer to a language that capitalises
differently or not at all. Feature ablation is useful precisely because it makes
that dependency visible instead of leaving it buried in learned weights.

</details>